# Workshop Databricks - Logística

Caso da **Atlântica Cargo**: entregas por caminhão, cabotagem, barco, balsa e ferrovia. O maior custo da empresa é logística. Objetivo: **atender a demanda no menor custo**, com OTIF e ocupação visíveis.

| Módulo | Resultado |
|---|---|
|1. Tabelas SQL | Três tabelas no Unity Catalog |
|2. Metric view | Camada semântica de custo e serviço |
|3. AI/BI + Genie Code | Dashboard com KPIs e mapas |
|4. Genie Agent | Sala curada, instruções e exemplos |
|5. Genie One | Mesmas perguntas, outra experiência |



## Modelo de dados

O workshop usa **exatamente três tabelas**. Entregas e rotas se ligam a `hubs_geo`.

```text
hubs_geo (hub_id)
   ├──< entregas (hub_origem_id, hub_destino_id)
   └──< rotas_entrega (hub_origem_id, hub_destino_id)
```

Operadora fictícia: **Atlântica Cargo**. Parceiros fictícios: VerdeMar Shipping, RioNorte Express, SerraSul Cargas, CostaBrava Log. Clientes fictícios: Casa Norte Atacado, Mercado Vale Verde, Indústria Brisa Alimentos, Farmácia Sol Nascente, Construtora Horizonte, Rede SuperLeste.

---

### `hubs_geo`
Cadastro de CDs, portos e bases, com capacidade e localização.

| Coluna | Tipo | Descrição |
|---|---|---|
| `hub_id` | STRING | Identificador único do hub (ex.: `HUB-SP-02`). Chave primária. |
| `hub_nome` | STRING | Nome fictício do hub. |
| `tipo_hub` | STRING | `Centro de distribuição`, `Porto`, `Base fluvial`, `Terminal ferroviário` ou `Terminal rodoviário`. |
| `operadora` | STRING | Marca fictícia que opera o hub. |
| `regiao` | STRING | Macroregião: `Norte`, `Nordeste`, `Centro-Oeste`, `Sudeste` ou `Sul`. |
| `estado_nome` | STRING | Nome do estado. |
| `estado_sigla` | STRING | UF (ex.: `SP`, `AM`). |
| `cidade` | STRING | Cidade do hub. |
| `pais` | STRING | País da localidade. |
| `latitude` | DOUBLE | Latitude WGS84 em graus decimais. |
| `longitude` | DOUBLE | Longitude WGS84 em graus decimais. |
| `capacidade_ton_dia` | DOUBLE | Capacidade nominal, em **t/dia**. |
| `custo_fixo_diario_brl` | DOUBLE | Custo fixo diário do hub, em **R$**. |
| `data_abertura` | DATE | Data de inauguração. |

---

### `entregas`
Uma linha = uma entrega. Fato de custo, demanda e serviço.

| Coluna | Tipo | Descrição |
|---|---|---|
| `entrega_id` | STRING | Identificador único da entrega. |
| `data_entrega` | TIMESTAMP | Data e hora da movimentação. |
| `hub_origem_id` | STRING | Hub de origem (`hubs_geo.hub_id`). |
| `hub_destino_id` | STRING | Hub de destino (`hubs_geo.hub_id`). |
| `rota_id` | STRING | Corredor multimodal (ex.: `RTA-SP-01-RJ-01-CAM`). |
| `modo_transporte` | STRING | `Caminhão`, `Cabotagem`, `Barco`, `Balsa` ou `Ferrovia`. |
| `operadora` | STRING | Marca fictícia responsável pelo trecho. |
| `volume_ton` | DOUBLE | Volume movimentado, em **toneladas**. Pode somar. |
| `demanda_unidades` | INT | Demanda em unidades/caixas. Pode somar. |
| `custo_frete_brl` | DOUBLE | Custo de frete, em **R$**. |
| `custo_combustivel_brl` | DOUBLE | Custo de combustível, em **R$**. |
| `custo_pedagio_brl` | DOUBLE | Pedágio ou taxa equivalente, em **R$**. |
| `custo_total_brl` | DOUBLE | Soma dos custos da entrega, em **R$**. |
| `distancia_km` | DOUBLE | Distância do trecho, em **km**. |
| `tempo_horas` | DOUBLE | Tempo de trânsito, em **horas**. Use média, não soma, se a pergunta for “prazo médio”. |
| `ocupacao_pct` | DOUBLE | Ocupação do veículo/navio de **0 a 100**. **Nunca some; use média.** |
| `status_entrega` | STRING | `No prazo`, `Atrasada`, `Parcial` ou `Cancelada`. |
| `entregue_no_prazo` | BOOLEAN | `true` se OTIF daquela entrega. |
| `cliente_marca` | STRING | Marca fictícia do embarcador. |

---

### `rotas_entrega`
Pontos ordenados dos corredores, para path map.

| Coluna | Tipo | Descrição |
|---|---|---|
| `ponto_id` | STRING | Identificador único do ponto. |
| `rota_id` | STRING | Identificador do corredor. |
| `ordem_ponto` | INTEGER | Sequência crescente dos pontos na rota. |
| `ponto_ts` | TIMESTAMP | Horário ilustrativo no trajeto. |
| `hub_origem_id` | STRING | Origem do corredor. |
| `hub_destino_id` | STRING | Destino do corredor. |
| `hub_proximo_id` | STRING | Hub mais próximo naquele trecho. |
| `modo_transporte` | STRING | Modo do corredor. |
| `operadora` | STRING | Operadora do corredor. |
| `latitude` | DOUBLE | Latitude WGS84 do ponto. |
| `longitude` | DOUBLE | Longitude WGS84 do ponto. |
| `distancia_acumulada_km` | DOUBLE | Distância acumulada desde a origem, em km. |
| `status_rota` | STRING | Situação do corredor (`Ativa`). |


---
## Configuração inicial

Preencha os widgets com o catálogo, o schema e um prefixo individual. Exemplo de prefixo: `carol`.

As tabelas serão criadas como `{catalogo}.{schema}.{prefixo}_hubs_geo`.

### Como copiar o caminho da pasta `dados`

1. No painel à esquerda do notebook, clique no ícone de pasta (**Workspace**) para ver os arquivos do workshop.
2. Encontre a pasta **`dados`**, ao lado de `notebook_guia_workshop` e `README.md`.
3. Clique com o botão direito na pasta **`dados`** (ou nos três pontinhos ao lado dela).
4. Escolha **Copy URL/path > Full path**. Não use **URL**: ela copia o link do navegador, não o caminho.
5. Cole o valor no widget **Caminho da pasta dados** (`caminho_dados`), no topo do notebook.

O caminho copiado tem este formato:

```text
/Workspace/Users/seu_usuario@seu_email.com/workshop-genie-aibi-logistica/dados
```

<div style="padding: 15px; border-left: 5px solid #d9534f; background-color: #fdf7f7; color: #a94442;">
<strong>🛑 ATENÇÃO:</strong> copie o caminho da pasta <code>dados</code>, não do notebook. O caminho precisa terminar em <code>/dados</code>. Depois de colar, rode a célula dos widgets abaixo e confira a linha <code>Dados:</code> no resultado.
</div>


In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.removeAll()
dbutils.widgets.text("nome_catalogo", "workspace", "Catálogo")
dbutils.widgets.text("nome_schema", "workshop_logistica", "Schema")
dbutils.widgets.text("seu_prefixo", "prefix_", "Seu prefixo")
dbutils.widgets.text(
    "caminho_dados",
    "/Workspace/Users/{seu_usuario}/workshop-genie-aibi-logistica/dados",
    "Caminho da pasta dados",
)

nome_catalogo = dbutils.widgets.get("nome_catalogo").strip()
nome_schema = dbutils.widgets.get("nome_schema").strip()
seu_prefixo = dbutils.widgets.get("seu_prefixo").strip()
caminho_dados = dbutils.widgets.get("caminho_dados").strip().rstrip("/")

assert nome_catalogo, "Informe o catálogo"
assert nome_schema, "Informe o schema"
assert seu_prefixo, "Informe um prefixo individual"
assert caminho_dados, "Informe o caminho da pasta dados"
print(f"Destino: {nome_catalogo}.{nome_schema} | Prefixo: {seu_prefixo}")
print(f"Dados: {caminho_dados}")


---
## Módulo 1 — Criar tabelas com SQL

Execute as células abaixo na ordem. Elas usam os widgets (`${nome_catalogo}`, `${nome_schema}`, `${seu_prefixo}`, `${caminho_dados}`) nas queries.

Tabelas criadas:

| CSV | Tabela |
|---|---|
| `hubs_geo.csv` | `{seu_prefixo}_hubs_geo` |
| `entregas.csv` | `{seu_prefixo}_entregas` |
| `rotas_entrega.csv` | `{seu_prefixo}_rotas_entrega` |

Se o `read_files` falhar, use a alternativa manual no final deste módulo.


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS `${nome_catalogo}`.`${nome_schema}`;


In [0]:
%sql
CREATE OR REPLACE TABLE `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_hubs_geo` (
  hub_id STRING COMMENT 'Identificador único do hub',
  hub_nome STRING COMMENT 'Nome fictício do hub',
  tipo_hub STRING COMMENT 'CD, porto, base fluvial ou terminal',
  operadora STRING COMMENT 'Marca fictícia que opera o hub',
  regiao STRING COMMENT 'Macroregião do hub',
  estado_nome STRING COMMENT 'Nome do estado',
  estado_sigla STRING COMMENT 'UF do estado',
  cidade STRING COMMENT 'Cidade do hub',
  pais STRING COMMENT 'País da localidade',
  latitude DOUBLE COMMENT 'Latitude WGS84 em graus decimais',
  longitude DOUBLE COMMENT 'Longitude WGS84 em graus decimais',
  capacidade_ton_dia DOUBLE COMMENT 'Capacidade nominal em toneladas por dia',
  custo_fixo_diario_brl DOUBLE COMMENT 'Custo fixo diário do hub em R$',
  data_abertura DATE COMMENT 'Data de inauguração do hub'
)
COMMENT 'Cadastro de hubs da Atlântica Cargo, com capacidade e localização';

INSERT OVERWRITE TABLE `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_hubs_geo`
SELECT
  CAST(hub_id AS STRING) AS hub_id,
  CAST(hub_nome AS STRING) AS hub_nome,
  CAST(tipo_hub AS STRING) AS tipo_hub,
  CAST(operadora AS STRING) AS operadora,
  CAST(regiao AS STRING) AS regiao,
  CAST(estado_nome AS STRING) AS estado_nome,
  CAST(estado_sigla AS STRING) AS estado_sigla,
  CAST(cidade AS STRING) AS cidade,
  CAST(pais AS STRING) AS pais,
  CAST(latitude AS DOUBLE) AS latitude,
  CAST(longitude AS DOUBLE) AS longitude,
  CAST(capacidade_ton_dia AS DOUBLE) AS capacidade_ton_dia,
  CAST(custo_fixo_diario_brl AS DOUBLE) AS custo_fixo_diario_brl,
  CAST(data_abertura AS DATE) AS data_abertura
FROM read_files(
  '${caminho_dados}',
  format => 'csv',
  header => true,
  inferSchema => true,
  pathGlobFilter => 'hubs_geo.csv'
);


In [0]:
%sql
CREATE OR REPLACE TABLE `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_entregas` (
  entrega_id STRING COMMENT 'Identificador único da entrega',
  data_entrega TIMESTAMP COMMENT 'Data e hora da movimentação',
  hub_origem_id STRING COMMENT 'Hub de origem, referência a hubs_geo',
  hub_destino_id STRING COMMENT 'Hub de destino, referência a hubs_geo',
  rota_id STRING COMMENT 'Corredor multimodal',
  modo_transporte STRING COMMENT 'Caminhão, Cabotagem, Barco, Balsa ou Ferrovia',
  operadora STRING COMMENT 'Marca fictícia responsável pelo trecho',
  volume_ton DOUBLE COMMENT 'Volume em toneladas. Pode somar',
  demanda_unidades INT COMMENT 'Demanda em unidades ou caixas. Pode somar',
  custo_frete_brl DOUBLE COMMENT 'Custo de frete em R$',
  custo_combustivel_brl DOUBLE COMMENT 'Custo de combustível em R$',
  custo_pedagio_brl DOUBLE COMMENT 'Pedágio ou taxa equivalente em R$',
  custo_total_brl DOUBLE COMMENT 'Custo total da entrega em R$',
  distancia_km DOUBLE COMMENT 'Distância do trecho em km',
  tempo_horas DOUBLE COMMENT 'Tempo de trânsito em horas. Para prazo médio, use AVG',
  ocupacao_pct DOUBLE COMMENT 'Ocupação de 0 a 100. Usar média, nunca soma',
  status_entrega STRING COMMENT 'No prazo, Atrasada, Parcial ou Cancelada',
  entregue_no_prazo BOOLEAN COMMENT 'true se a entrega entrou no OTIF',
  cliente_marca STRING COMMENT 'Marca fictícia do embarcador'
)
COMMENT 'Entregas multimodais: demanda, custo, prazo e ocupação';

INSERT OVERWRITE TABLE `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_entregas`
SELECT
  CAST(entrega_id AS STRING) AS entrega_id,
  CAST(data_entrega AS TIMESTAMP) AS data_entrega,
  CAST(hub_origem_id AS STRING) AS hub_origem_id,
  CAST(hub_destino_id AS STRING) AS hub_destino_id,
  CAST(rota_id AS STRING) AS rota_id,
  CAST(modo_transporte AS STRING) AS modo_transporte,
  CAST(operadora AS STRING) AS operadora,
  CAST(volume_ton AS DOUBLE) AS volume_ton,
  CAST(demanda_unidades AS INT) AS demanda_unidades,
  CAST(custo_frete_brl AS DOUBLE) AS custo_frete_brl,
  CAST(custo_combustivel_brl AS DOUBLE) AS custo_combustivel_brl,
  CAST(custo_pedagio_brl AS DOUBLE) AS custo_pedagio_brl,
  CAST(custo_total_brl AS DOUBLE) AS custo_total_brl,
  CAST(distancia_km AS DOUBLE) AS distancia_km,
  CAST(tempo_horas AS DOUBLE) AS tempo_horas,
  CAST(ocupacao_pct AS DOUBLE) AS ocupacao_pct,
  CAST(status_entrega AS STRING) AS status_entrega,
  CAST(entregue_no_prazo AS BOOLEAN) AS entregue_no_prazo,
  CAST(cliente_marca AS STRING) AS cliente_marca
FROM read_files(
  '${caminho_dados}',
  format => 'csv',
  header => true,
  inferSchema => true,
  pathGlobFilter => 'entregas.csv'
);


In [0]:
%sql
CREATE OR REPLACE TABLE `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_rotas_entrega` (
  ponto_id STRING COMMENT 'Identificador único do ponto na rota',
  rota_id STRING COMMENT 'Identificador do corredor',
  ordem_ponto INT COMMENT 'Sequência crescente dos pontos dentro da rota',
  ponto_ts TIMESTAMP COMMENT 'Horário ilustrativo no trajeto',
  hub_origem_id STRING COMMENT 'Origem do corredor, referência a hubs_geo',
  hub_destino_id STRING COMMENT 'Destino do corredor, referência a hubs_geo',
  hub_proximo_id STRING COMMENT 'Hub mais próximo naquele trecho',
  modo_transporte STRING COMMENT 'Modo do corredor',
  operadora STRING COMMENT 'Operadora do corredor',
  latitude DOUBLE COMMENT 'Latitude WGS84 do ponto',
  longitude DOUBLE COMMENT 'Longitude WGS84 do ponto',
  distancia_acumulada_km DOUBLE COMMENT 'Distância acumulada desde a origem, em km',
  status_rota STRING COMMENT 'Situação do corredor'
)
COMMENT 'Pontos ordenados de rotas multimodais para path map';

INSERT OVERWRITE TABLE `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_rotas_entrega`
SELECT
  CAST(ponto_id AS STRING) AS ponto_id,
  CAST(rota_id AS STRING) AS rota_id,
  CAST(ordem_ponto AS INT) AS ordem_ponto,
  CAST(ponto_ts AS TIMESTAMP) AS ponto_ts,
  CAST(hub_origem_id AS STRING) AS hub_origem_id,
  CAST(hub_destino_id AS STRING) AS hub_destino_id,
  CAST(hub_proximo_id AS STRING) AS hub_proximo_id,
  CAST(modo_transporte AS STRING) AS modo_transporte,
  CAST(operadora AS STRING) AS operadora,
  CAST(latitude AS DOUBLE) AS latitude,
  CAST(longitude AS DOUBLE) AS longitude,
  CAST(distancia_acumulada_km AS DOUBLE) AS distancia_acumulada_km,
  CAST(status_rota AS STRING) AS status_rota
FROM read_files(
  '${caminho_dados}',
  format => 'csv',
  header => true,
  inferSchema => true,
  pathGlobFilter => 'rotas_entrega.csv'
);


In [0]:
%sql
SELECT 'hubs_geo' AS tabela, COUNT(*) AS linhas FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_hubs_geo`
UNION ALL
SELECT 'entregas', COUNT(*) FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_entregas`
UNION ALL
SELECT 'rotas_entrega', COUNT(*) FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_rotas_entrega`;


---
### Alternativa — upload manual dos CSVs

Use este caminho só se o SQL com `read_files` não funcionar.

1. Abra **+ New > Add or upload data** ou **Catalog > Create > Create table**.
2. Faça o download dos CSVs da pasta `dados/` e faça o upload de cada arquivo, um por um.
3. Revise os tipos detectados, principalmente timestamps, booleanos, latitude e longitude.
4. Selecione o mesmo catálogo e schema dos widgets.
5. Crie as tabelas com estes nomes. Não esqueça o **prefixo_**.

| Arquivo | Nome da tabela |
|---|---|
| `hubs_geo.csv` | `{seu_prefixo}_hubs_geo` |
| `entregas.csv` | `{seu_prefixo}_entregas` |
| `rotas_entrega.csv` | `{seu_prefixo}_rotas_entrega` |


---
## Módulo 2 — Metric view (camada semântica)

### O que é uma metric view

Uma **metric view** é um objeto do Unity Catalog que guarda a **definição oficial das métricas de negócio**, como "custo total" ou "ocupação média". Funciona como uma camada semântica: você define a métrica uma vez e todas as ferramentas (SQL, dashboards AI/BI, Genie) usam a mesma regra.

Duas partes:

- **Dimensões** (*fields*): período, modo, operadora, origem/destino, `modal_alternativo`, faixas de distância e ocupação.
- **Medidas** (*measures*): `custo_total`, `otif`, `ocupacao_media`, `custo_por_tonelada` (razão de somas).

**Diferença para uma view comum:** a view já vem com `GROUP BY` fixo. Na metric view a agregação só acontece na consulta: você escolhe as dimensões e o Databricks calcula a medida para aquele recorte.

**Por que isso importa aqui:** `ocupacao_media` nunca pode ser somada, e custo por tonelada **não** é a média das razões por linha — é a razão das somas. Na metric view isso fica travado.

### Criar com Genie Code / Assistant

1. Abra o **SQL Editor** (ou Catalog Explorer > Create > Metric view) e o **Genie / Assistant**.
2. Rode a célula Python abaixo e copie o prompt já preenchido.
3. Cole no chat. Peça para executar o `CREATE OR REPLACE VIEW ... WITH METRICS`.
4. Se o agente não executar DDL, use a célula SQL de fallback mais abaixo.

O mesmo texto está em `prompts/01_metric_view.md`.

<div style="padding: 15px; border-left: 5px solid #d9534f; background-color: #fdf7f7; color: #a94442;">
<strong>🛑 ATENÇÃO:</strong> rode a célula para substituir catálogo, schema e prefixo. Sem isso o agente cria o objeto no lugar errado.
</div>


In [0]:
nome_catalogo = dbutils.widgets.get("nome_catalogo").strip()
nome_schema = dbutils.widgets.get("nome_schema").strip()
seu_prefixo = dbutils.widgets.get("seu_prefixo").strip()

entregas = f"{nome_catalogo}.{nome_schema}.{seu_prefixo}_entregas"
hubs = f"{nome_catalogo}.{nome_schema}.{seu_prefixo}_hubs_geo"
mv = f"{seu_prefixo}_mv_logistica"

prompt_metric_view = f"""
Crie uma metric view no Unity Catalog chamada {mv}, no mesmo schema de @{entregas}, com base nessa tabela.
Faça dois joins many-to-one com @{hubs}: um como "origem" (hub_origem_id = hub_id) e outro como "destino" (hub_destino_id = hub_id).

Dimensões:
- periodo (mês de referência, a partir de data_entrega)
- ano e mes (derivados do período)
- modo_transporte
- operadora
- status_entrega
- cliente_marca
- rota_id
- hub_origem (nome do hub de origem) e hub_destino (nome do hub de destino)
- regiao_origem, estado_origem e cidade_origem
- estado_destino e cidade_destino
- modal_alternativo (Sim quando o modo for Cabotagem, Barco, Balsa ou Ferrovia; Não quando for Caminhão)
- faixa_distancia (até 500 km, 500 a 1.000, 1.000 a 2.000, acima de 2.000, com base em distancia_km)
- faixa_ocupacao (abaixo de 60%, 60 a 75%, 75 a 90%, acima de 90%, com base em ocupacao_pct)

Métricas:
- entregas: quantidade de entregas
- entregas_no_prazo: quantidade de entregas com entregue_no_prazo verdadeiro (nulo é tratado como falso)
- otif: entregas_no_prazo dividido por entregas, em percentual
- entregas_atrasadas: quantidade de entregas com status Atrasada
- taxa_atraso: entregas_atrasadas dividido por entregas, em percentual
- volume_total: soma de volume_ton
- demanda_total: soma de demanda_unidades
- custo_total: soma de custo_total_brl, tratando nulo como zero
- custo_combustivel: soma de custo_combustivel_brl, tratando nulo como zero
- participacao_combustivel: custo_combustivel dividido por custo_total, em percentual
- custo_por_tonelada: custo_total dividido por volume_total
- custo_por_km: custo_total dividido pela soma de distancia_km
- ocupacao_media: média de ocupacao_pct, desconsiderando entregas com status Cancelada
- tempo_medio_transito: média de tempo_horas
- percentual_volume_modal_alternativo: percentual do volume_total transportado fora do Caminhão
- mediana_custo_entrega: mediana de custo_total_brl

Regras:
- Use NULLIF no denominador de todas as divisões para evitar divisão por zero.
- Razões (custo_por_tonelada, custo_por_km, otif, taxa_atraso) devem ser razão de somas, nunca média de razões por linha.
- ocupacao_media e otif nunca podem ser somados.

Adicione comentários em português em cada dimensão e métrica, explicando a regra de negócio. Inclua display_name e synonyms em português (ex.: modal, frete, R$/t, nível de serviço).

Após criar, valide executando custo_por_tonelada, otif e custo_total por modo_transporte nos últimos 3 meses e apresente o resultado.
""".strip()

print(prompt_metric_view)

### Fallback SQL — criar a metric view no notebook

Use se o Genie Code não executar o DDL. SQL Warehouse / Runtime **17.3+** recomendado.


In [0]:
nome_catalogo = dbutils.widgets.get("nome_catalogo").strip()
nome_schema = dbutils.widgets.get("nome_schema").strip()
seu_prefixo = dbutils.widgets.get("seu_prefixo").strip()

sql = f"""
CREATE OR REPLACE VIEW `{nome_catalogo}`.`{nome_schema}`.`{seu_prefixo}_mv_logistica`
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: KPIs da malha Atlântica Cargo. Custo, demanda, OTIF, ocupação e R$/t. Razões são sempre soma/soma.

source: {nome_catalogo}.{nome_schema}.{seu_prefixo}_entregas

joins:
  - name: origem
    source: {nome_catalogo}.{nome_schema}.{seu_prefixo}_hubs_geo
    on: source.hub_origem_id = origem.hub_id
    cardinality: many_to_one
    rely:
      at_most_one_match: true
  - name: destino
    source: {nome_catalogo}.{nome_schema}.{seu_prefixo}_hubs_geo
    on: source.hub_destino_id = destino.hub_id
    cardinality: many_to_one
    rely:
      at_most_one_match: true

fields:
  - name: periodo
    expr: DATE_TRUNC('MONTH', source.data_entrega)
    display_name: Período
    comment: Mês de referência da entrega, derivado de data_entrega.
    synonyms: [mês, competência, mês de referência]
  - name: ano
    expr: YEAR(source.data_entrega)
    display_name: Ano
    comment: Ano calendário derivado do período da entrega.
  - name: mes
    expr: MONTH(source.data_entrega)
    display_name: Mês
    comment: Número do mês (1 a 12) derivado do período da entrega.
  - name: modo_transporte
    expr: source.modo_transporte
    display_name: Modo de transporte
    comment: Modal da entrega. Valores Caminhão, Cabotagem, Barco, Balsa ou Ferrovia.
    synonyms: [modal, caminhão, barco, cabotagem, balsa, ferrovia]
  - name: operadora
    expr: source.operadora
    display_name: Operadora
    comment: Marca fictícia responsável pelo trecho.
    synonyms: [transportadora, parceiro]
  - name: status_entrega
    expr: source.status_entrega
    display_name: Status da entrega
    comment: No prazo, Atrasada, Parcial ou Cancelada.
  - name: cliente_marca
    expr: source.cliente_marca
    display_name: Cliente
    comment: Marca fictícia do embarcador.
  - name: rota_id
    expr: source.rota_id
    display_name: Corredor
    comment: Identificador do corredor multimodal origem-destino-modo.
  - name: hub_origem
    expr: origem.hub_nome
    display_name: Hub de origem
    comment: Nome do hub de origem (join many-to-one em hubs_geo).
  - name: hub_destino
    expr: destino.hub_nome
    display_name: Hub de destino
    comment: Nome do hub de destino (join many-to-one em hubs_geo).
  - name: regiao_origem
    expr: origem.regiao
    display_name: Região de origem
    comment: Macroregião do hub de origem.
  - name: estado_origem
    expr: origem.estado_nome
    display_name: Estado de origem
    comment: Nome do estado do hub de origem. Usar este campo em choropleth, não a sigla.
    synonyms: [UF origem, unidade federativa origem]
  - name: cidade_origem
    expr: origem.cidade
    display_name: Cidade de origem
    comment: Cidade do hub de origem.
  - name: estado_destino
    expr: destino.estado_nome
    display_name: Estado de destino
    comment: Nome do estado do hub de destino.
  - name: cidade_destino
    expr: destino.cidade
    display_name: Cidade de destino
    comment: Cidade do hub de destino.
  - name: modal_alternativo
    expr: CASE WHEN source.modo_transporte = 'Caminhão' THEN 'Não' ELSE 'Sim' END
    display_name: Modal alternativo
    comment: Sim quando o modo é Cabotagem, Barco, Balsa ou Ferrovia. Não quando é Caminhão. Alavanca de custo da malha.
  - name: faixa_distancia
    expr: CASE WHEN source.distancia_km <= 500 THEN 'até 500 km' WHEN source.distancia_km <= 1000 THEN '500 a 1.000' WHEN source.distancia_km <= 2000 THEN '1.000 a 2.000' ELSE 'acima de 2.000' END
    display_name: Faixa de distância
    comment: Faixa de distancia_km da entrega. Cortes em 500, 1.000 e 2.000 km.
  - name: faixa_ocupacao
    expr: CASE WHEN source.ocupacao_pct < 60 THEN 'abaixo de 60%' WHEN source.ocupacao_pct < 75 THEN '60 a 75%' WHEN source.ocupacao_pct < 90 THEN '75 a 90%' ELSE 'acima de 90%' END
    display_name: Faixa de ocupação
    comment: Faixa de ocupacao_pct da entrega. Cortes em 60, 75 e 90.

measures:
  - name: entregas
    expr: COUNT(1)
    display_name: Entregas
    comment: Quantidade de entregas no recorte. Inclui canceladas.
    synonyms: [quantidade, volume de pedidos]
  - name: entregas_no_prazo
    expr: SUM(CASE WHEN COALESCE(source.entregue_no_prazo, false) THEN 1 ELSE 0 END)
    display_name: Entregas no prazo
    comment: Quantidade de entregas com entregue_no_prazo verdadeiro. Nulo é tratado como falso.
  - name: otif
    expr: 100 * SUM(CASE WHEN COALESCE(source.entregue_no_prazo, false) THEN 1 ELSE 0 END) / NULLIF(COUNT(1), 0)
    display_name: OTIF (%)
    comment: entregas_no_prazo dividido por entregas, em percentual. Razão de somas. Nunca somar este percentual.
    synonyms: [no prazo, nível de serviço, pontualidade]
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
  - name: entregas_atrasadas
    expr: SUM(CASE WHEN source.status_entrega = 'Atrasada' THEN 1 ELSE 0 END)
    display_name: Entregas atrasadas
    comment: Quantidade de entregas com status Atrasada.
  - name: taxa_atraso
    expr: 100 * SUM(CASE WHEN source.status_entrega = 'Atrasada' THEN 1 ELSE 0 END) / NULLIF(COUNT(1), 0)
    display_name: Taxa de atraso (%)
    comment: entregas_atrasadas dividido por entregas, em percentual. Razão de somas.
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
  - name: volume_total
    expr: SUM(source.volume_ton)
    display_name: Volume total (t)
    comment: Soma de volume_ton. Pode somar.
    synonyms: [tonelada, volume, demanda em t]
  - name: demanda_total
    expr: SUM(source.demanda_unidades)
    display_name: Demanda (unidades)
    comment: Soma de demanda_unidades (caixas/unidades).
  - name: custo_total
    expr: SUM(COALESCE(source.custo_total_brl, 0))
    display_name: Custo total (R$)
    comment: Soma de custo_total_brl, tratando nulo como zero.
    synonyms: [custo, gasto logístico, frete total]
  - name: custo_combustivel
    expr: SUM(COALESCE(source.custo_combustivel_brl, 0))
    display_name: Custo de combustível (R$)
    comment: Soma de custo_combustivel_brl, tratando nulo como zero.
  - name: participacao_combustivel
    expr: 100 * SUM(COALESCE(source.custo_combustivel_brl, 0)) / NULLIF(SUM(COALESCE(source.custo_total_brl, 0)), 0)
    display_name: Participação do combustível (%)
    comment: custo_combustivel dividido por custo_total, em percentual. Razão de somas.
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
  - name: custo_por_tonelada
    expr: SUM(COALESCE(source.custo_total_brl, 0)) / NULLIF(SUM(source.volume_ton), 0)
    display_name: Custo por tonelada (R$/t)
    comment: custo_total dividido por volume_total. Razão de somas, nunca média de R$/t por linha.
    synonyms: [R$/t, custo unitário, eficiência de custo]
  - name: custo_por_km
    expr: SUM(COALESCE(source.custo_total_brl, 0)) / NULLIF(SUM(source.distancia_km), 0)
    display_name: Custo por km (R$/km)
    comment: custo_total dividido pela soma de distancia_km. Razão de somas.
  - name: ocupacao_media
    expr: AVG(CASE WHEN source.status_entrega <> 'Cancelada' THEN source.ocupacao_pct END)
    display_name: Ocupação média (%)
    comment: Média de ocupacao_pct, desconsiderando entregas Cancelada (ocupação 0 puxaria a média). Nunca somar.
    synonyms: [ocupação, fill rate, aproveitamento]
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
  - name: tempo_medio_transito
    expr: AVG(source.tempo_horas)
    display_name: Tempo médio de trânsito (h)
    comment: Média de tempo_horas. Nunca somar se a pergunta for prazo médio.
  - name: percentual_volume_modal_alternativo
    expr: 100 * SUM(CASE WHEN source.modo_transporte <> 'Caminhão' THEN source.volume_ton ELSE 0 END) / NULLIF(SUM(source.volume_ton), 0)
    display_name: Volume em modal alternativo (%)
    comment: Percentual do volume_total transportado fora do Caminhão (cabotagem, barco, balsa, ferrovia).
    format:
      type: number
      decimal_places:
        type: exact
        places: 2
  - name: mediana_custo_entrega
    expr: MEDIAN(source.custo_total_brl)
    display_name: Mediana do custo da entrega (R$)
    comment: Mediana de custo_total_brl. Não confundir com a média nem com o custo_total.
$$
"""

spark.sql(sql)
print(f"Metric view `{nome_catalogo}`.`{nome_schema}`.`{seu_prefixo}_mv_logistica` criada com sucesso!")


### Validar a metric view

Toda medida precisa de `MEASURE(...)`. A célula abaixo replica a validação pedida no prompt do Genie Code: `custo_por_tonelada`, `otif` e `custo_total` por `modo_transporte` nos últimos 3 meses.


In [0]:
%sql
SELECT
  modo_transporte,
  MEASURE(`custo_total`) AS custo_total,
  MEASURE(`custo_por_tonelada`) AS custo_por_tonelada,
  MEASURE(`otif`) AS otif,
  MEASURE(`ocupacao_media`) AS ocupacao_media,
  MEASURE(`entregas`) AS entregas
FROM `${nome_catalogo}`.`${nome_schema}`.`${seu_prefixo}_mv_logistica`
WHERE periodo >= ADD_MONTHS(DATE_TRUNC('MONTH', CURRENT_DATE()), -3)
GROUP BY ALL
ORDER BY custo_total DESC
;


### Visite a metric view no Catalog Explorer

1. No menu lateral, clique em **Catalog**.
2. Procure `{seu_prefixo}_mv_logistica`.
3. O tipo do objeto é **Metric view**, não tabela.
4. Clique em **Edit**. Na aba **UI**, abra as medidas:
   - `custo_total` usa `SUM` (nulo = 0);
   - `ocupacao_media` usa `AVG` e ignora `Cancelada`;
   - `custo_por_tonelada` e `otif` são razão de somas (`NULLIF` no denominador);
   - `mediana_custo_entrega` usa `MEDIAN`.
5. Abra as dimensões (`periodo`, `modal_alternativo`, `faixa_distancia`, `faixa_ocupacao`, origem/destino).
6. Clique em `<>` para ver o YAML. Saia **sem salvar**.

Pergunta para a sala: se você consultar R$/t só por `modo_transporte` e depois por `estado_origem` e `modo_transporte`, por que o valor muda mas a regra de cálculo continua a mesma?


---
## Módulo 3 — AI/BI com Genie Code

1. Abra **Dashboards > Create dashboard**.
2. Abra Genie Code.
3. Referencie com `@` a metric view `{seu_prefixo}_mv_logistica` e as tabelas `{seu_prefixo}_hubs_geo` e `{seu_prefixo}_rotas_entrega`.
4. Rode a célula Python abaixo e copie o prompt. Ele pede KPIs da metric view e **três mapas** (choropleth, point e path).

O mesmo texto está em `prompts/02_aibi_dashboard.md`.

<div style="padding: 15px; border-left: 5px solid #d9534f; background-color: #fdf7f7; color: #a94442;">
<strong>🛑 ATENÇÃO:</strong> rode a célula para substituir catálogo, schema e prefixo. Sem isso o Genie Code não encontra os objetos.
</div>


In [0]:
nome_catalogo = dbutils.widgets.get("nome_catalogo").strip()
nome_schema = dbutils.widgets.get("nome_schema").strip()
seu_prefixo = dbutils.widgets.get("seu_prefixo").strip()

def obj(sufixo):
    return f"`{nome_catalogo}`.`{nome_schema}`.`{seu_prefixo}_{sufixo}`"

mv = obj("mv_logistica")
hubs = obj("hubs_geo")
rotas = obj("rotas_entrega")

prompt_dashboard = f"""
Crie um dashboard chamado {seu_prefixo} - Atlântica Cargo — Custo e Rota usando a metric view de KPIs e as tabelas geográficas.

Fontes (obrigatório):
- Metric view {mv} para TODOS os KPIs, barras, linhas, heatmap e tabelas de custo/demanda/OTIF/ocupação. No AI/BI, MEASURE() é aplicado automaticamente. Use os nomes das medidas da metric view: custo_total, volume_total, demanda_total, custo_por_tonelada, otif, ocupacao_media, taxa_atraso, entregas, entregas_atrasadas, participacao_combustivel, mediana_custo_entrega.
- Dimensões úteis da metric view: periodo, modo_transporte, operadora, status_entrega, modal_alternativo, faixa_distancia, faixa_ocupacao, estado_origem, hub_origem, hub_destino, regiao_origem.
- Tabela {hubs} apenas para o point map (latitude, longitude, capacidade, tipo de hub).
- Tabela {rotas} apenas para o path map e para o gráfico de linha de distância acumulada (rota_id, ordem_ponto, latitude, longitude, modo_transporte, operadora, distancia_acumulada_km).

Não calcule custo, OTIF, ocupação ou custo por tonelada direto da tabela entregas. Se o gráfico for de KPI, a origem é a metric view.

Crie duas páginas. Evite gráfico de pizza. Títulos em português, unidades nos rótulos, moeda em R$ com 2 casas, percentuais com 2 casas.
Nos gráficos de ocupacao_media e de otif, comece o eixo Y em 50 (não em 0).

Página 1 — Visão executiva de custo e demanda
Faixa de KPIs no topo, depois uma grade:
- KPI: custo_total (R$)
- KPI: volume_total (t)
- KPI: custo_por_tonelada (R$/t)
- KPI: otif (%)
- KPI: ocupacao_media (%)
- Gráfico de linha: custo_total por periodo
- Gráfico de área: volume_total e demanda_total por periodo
- Gráfico de barras verticais: custo_total por modo_transporte
- Gráfico de barras horizontais: custo_por_tonelada por modo_transporte
- Gráfico combinado: barras de volume_total e linha de ocupacao_media por modo_transporte
- Gráfico de barras agrupadas: entregas vs entregas_atrasadas por operadora
- Tabela: ranking de corredores (hub_origem → hub_destino) com volume_total, custo_total, custo_por_tonelada, otif e ocupacao_media

Página 2 — Geografia da malha (mapas em destaque)
Três mapas, nesta ordem:

1. Choropleth por estado de ORIGEM: use regionType mapbox-v4-admin, admin0 com geographicRole admin0-iso-3166-1-alpha-3 e valor fixo "BRA" (não use admin0-name nem o valor "Brazil"), admin1 com geographicRole admin1-name e campo estado_origem. Cor = custo_total em R$, tooltip com volume_total (t), custo_por_tonelada e otif. Escala sequencial laranja/vermelho. Título: Custo logístico por estado de origem (R$).

2. Point map dos hubs: latitude e longitude de hubs_geo, tamanho por capacidade_ton_dia, cor por tipo_hub, tooltip com hub_nome, cidade, operadora e capacidade_ton_dia.

3. Path map chamado Rotas multimodais: para o dataset de geometria, use SQL que agrupa por rota_id e constrói a linha com ST_MAKELINE. Como ARRAY_AGG com ORDER BY não é suportado neste warehouse, use a seguinte lógica de ordenação: COLLECT_LIST de NAMED_STRUCT com campos ('seq', ordem_ponto, 'lng', longitude, 'lat', latitude), ordene com ARRAY_SORT passando um comparador lambda (l, r) -> CASE WHEN l.seq < r.seq THEN -1 WHEN l.seq > r.seq THEN 1 ELSE 0 END, depois aplique TRANSFORM para extrair ST_POINT(s.lng, s.lat), e envolva tudo em ST_MAKELINE. Armazene o resultado como coluna GEOMETRY (não aplique ST_ASGEOJSON no dataset). No widget, o campo da query deve usar a expressão ST_ASGEOJSON(`route_geom`) — o path-map exige que a expressão comece com ST_ASGEOJSON sobre uma coluna GEOMETRY; uma coluna STRING pré-computada é silenciosamente descartada. Cor por modo_transporte. Tooltip com rota_id, hub_origem_id, hub_destino_id, distancia_total_km.

Em seguida:
- Barras: custo_total por estado_origem
- Barras empilhadas (não 100%): volume_total por modo_transporte em cada regiao_origem
- Dispersão: custo_por_tonelada por faixa_distancia, cor por modo_transporte
- Heatmap: otif por modo_transporte e estado_origem
- Heatmap: custo_por_tonelada por faixa_distancia e faixa_ocupacao
- Linha: AVG(distancia_acumulada_km) ao longo de ordem_ponto, uma série por modo_transporte (dataset da tabela {seu_prefixo}_rotas_entrega, agregado)
- Tabela: hubs com hub_nome, cidade, tipo_hub, capacidade_ton_dia e custo_fixo_diario_brl

Regras dos mapas:
- rota_id separa os trajetos; nunca misture pontos de rotas diferentes
- ordem_ponto define a sequência do path
- não inverter latitude e longitude (longitude primeiro em ST_POINT)
- filtros globais não devem ser aplicados ao dataset do path map (usar dataset separado para rotas)
- choropleth usa nome completo do estado brasileiro (São Paulo, Amazonas), não a sigla

Filtros globais: modo_transporte, operadora, estado_origem, status_entrega, modal_alternativo.

Tema: fonte Poppins, cores consistentes por modo_transporte em todos os gráficos (Caminhão, Cabotagem, Barco, Balsa, Ferrovia).

Narrativa: atender a demanda no menor custo, mostrando onde o caminhão é caro, onde modal_alternativo = Sim dilui R$/t, e quais estados concentram custo e atraso.
""".strip()

print(prompt_dashboard)


### Usando uma imagem de referência para estilizar

1. Tire print de um site ou do brand deck do cliente (cores apenas; dados continuam sintéticos).
2. Cole a imagem no chat do Genie Code e peça:

> Estilize o dashboard seguindo como referência as cores da imagem em anexo.


### Publicar o dashboard e perguntar no Ask Genie

O rascunho é só para quem edita. Para outras pessoas perguntarem, publique e use **Ask Genie** na versão publicada.

#### Publicar

1. Clique em **Publish**.
2. **Share data permissions** (recomendado neste workshop): as queries rodam com as credenciais de quem publicou.
3. Deixe **Enable Genie** ligado.
4. Confirme **Publish**. Compartilhe com permissão de visualização.

#### Perguntar no Ask Genie (Genie do dashboard)

1. Abra a **versão publicada**.
2. Clique em **Ask Genie**.
3. Opcional: clique em um gráfico antes de perguntar.
4. Exemplos:
   - `Quais modos têm maior custo total e qual o R$/t de cada um?`
   - `Compare OTIF e ocupação média de Caminhão, Barco e Ferrovia.`
   - `Quais estados de origem concentram custo logístico?`
   - `Como está a eficiência?` — o Agent deve pedir esclarecimento.
   - `Gere um mapa do Brasil mostrando o volume total transportado por estado de origem`
   - `Crie um mapa de calor por estado mostrando a taxa de OTIF (entregas no prazo)`
   - `Mapeie o custo total de frete por estado de destino em escala de cores`
   - `Mostra a distribuição geográfica dos gastos logísticos`
   - `Crie um mapa do percentual de entregas atrasadas por estado de destino`
   - `Quais hubs ficam a menos de 500 km do hub de Vitória?`
5. Inspecione o SQL: KPI deve vir da metric view (`MEASURE`), não de `SUM(ocupacao_pct)`.

O Ask Genie responde com base nos datasets daquele dashboard. Para perguntas mais livres, use a sala do Módulo 4 ou o Genie One do Módulo 5.


---
## Módulo 4 — Genie Agent (15 min)

Trate o Genie como uma pessoa analista recém-chegada ao domínio:

1. **Comece pequeno:** metric view + duas tabelas geo.
2. **Documente tabelas e colunas:** granularidade e unidades (já no `COMMENT`).
3. **Prefira semântica estruturada:** metric view e exemplos SQL antes de instruções longas.
4. **Use exemplos SQL verificados:** ensine R$/t (razão de somas) e joins de origem/destino.
5. **Escreva instruções específicas:** gatilho, dado ausente, ação e pergunta de esclarecimento.
6. **Evite conflitos:** exemplos e instruções com as mesmas unidades.
7. **Teste o SQL gerado** antes de compartilhar.
8. **Itere** com perguntas reais do cliente.

### Criar o Genie Agent

1. Abra **Genie Agents > New**.
2. Adicione `{seu_prefixo}_mv_logistica`, `{seu_prefixo}_hubs_geo` e `{seu_prefixo}_rotas_entrega`. Só inclua `{seu_prefixo}_entregas` se quiser linha a linha.
3. Clique em **CREATE**.
4. **Configure** nome e descrição.

> Nome: Seu Nome - Atlântica Cargo — Workshop

> Descrição: Assistente para análise da malha logística multimodal da Atlântica Cargo. Explora custo, demanda, OTIF, ocupação, modos (caminhão, barco, cabotagem, balsa, ferrovia), rotas entre hubs e mapas da malha. Apoia trade-offs para atender a demanda no menor custo possível. Prefere resposta com mapa quando a pergunta for geográfica.

O prompt completo também está em `prompts/03_genie_agent.md`.


In [0]:
nome_catalogo = dbutils.widgets.get("nome_catalogo").strip()
nome_schema = dbutils.widgets.get("nome_schema").strip()
seu_prefixo = dbutils.widgets.get("seu_prefixo").strip()
mv = f"{nome_catalogo}.{nome_schema}.{seu_prefixo}_mv_logistica"

prompt_genie_agent = f"""
# Assistente de Logística — Atlântica Cargo

Você é um analista de redes de transporte. Responde perguntas sobre custo logístico, demanda atendida, eficiência de rota e nível de serviço (OTIF) da operação multimodal fictícia Atlântica Cargo e parceiros (VerdeMar Shipping, RioNorte Express, SerraSul Cargas, CostaBrava Log).

## Regras obrigatórias
- Responda em português.
- Sempre cite o período e as unidades (R$, toneladas, km, horas, %).
- Arredonde moeda e percentuais a duas casas.
- Use gráficos sempre que a pergunta comparar modos, estados ou o tempo.
- Sempre que for possível, responda com mapa geográfico (não só tabela ou barra). Se a pergunta citar estado, cidade, hub, Brasil, malha, distribuição, calor/heatmap espacial, proximidade em km ou destino/origem, o visual principal deve ser um mapa.
- KPI de custo, demanda, OTIF, ocupação, R$/t e R$/km → sempre a metric view `{mv}` e MEASURE().
- Use os nomes oficiais das medidas: custo_total, volume_total, demanda_total, custo_por_tonelada, custo_por_km, otif, ocupacao_media, taxa_atraso, entregas, entregas_atrasadas, mediana_custo_entrega, percentual_volume_modal_alternativo.
- Mapa de pontos / capacidade / distância entre hubs → hubs_geo (latitude, longitude).
- Choropleth por estado → estado_origem ou estado_destino com nome completo (São Paulo, Amazonas), nunca a sigla. País = Brasil; em Mapbox use admin0 ISO 3166-1 alpha-3 = BRA (não o valor "Brazil").
- Trajeto / path / ordem dos pontos → rotas_entrega, agrupando por rota_id e ordenando por ordem_ponto.
- Nunca some ocupacao_media nem otif. São médias ou taxas.
- custo_por_tonelada = custo_total / volume_total. Nunca média de (custo/volume) por linha.
- custo_por_km = custo_total / soma da distância.
- ocupacao_media ignora entregas Cancelada.

## Mapas — quando usar cada tipo
- Volume, custo, OTIF ou atraso por estado → choropleth.
- Localização de hubs, capacidade ou “quais hubs perto de X” → point map em hubs_geo. Distância em km a partir de latitude/longitude (haversine ou ST_DISTANCE). Se o hub pedido não existir no cadastro, diga isso e ofereça o hub mais próximo da cidade/estado.
- Rotas entre origem e destino → path map em rotas_entrega.
- “Distribuição geográfica dos gastos” → choropleth de custo_total por estado (origem, salvo se a pergunta pedir destino).

## Regras de negócio
- Uma linha em entregas é uma entrega.
- Modos: Caminhão, Cabotagem, Barco, Balsa, Ferrovia.
- modal_alternativo = Sim quando o modo não é Caminhão.
- Status: No prazo, Atrasada, Parcial, Cancelada.
- OTIF: entregas_no_prazo / entregas.
- Taxa de atraso: entregas_atrasadas / entregas (medida taxa_atraso).
- Origem e destino são hubs distintos.
- O maior custo da empresa é logística: ao sugerir "melhor rota", otimize custo_total e custo_por_tonelada sem piorar otif de forma material. Deixe o trade-off explícito (custo vs prazo vs ocupação).

## Quando pedir esclarecimento
- "melhor rota" sozinho → pergunte: menor custo, menor tempo, maior OTIF ou maior ocupação?
- "demanda" ambígua → pergunte: volume_total (toneladas) ou demanda_total (unidades)?
- tendência sem período → pergunte: Qual período deseja analisar?
- "eficiência" sozinha → pergunte: custo_por_tonelada, ocupacao_media ou percentual_volume_modal_alternativo?
- mapa sem métrica → pergunte: volume, custo, OTIF ou atraso? origem ou destino?
""".strip()

print(prompt_genie_agent)


### Examples na sala Genie

Na tela do Genie Agent, abra **Examples > Add > Example query**. Cada exemplo tem:

1. **What question does this query answer?**
2. **SQL**
3. **Parameters** — vazio nestes exemplos
4. **Usage Guidance**
5. **Preview**, depois **Save**

<div style="padding: 15px; border-left: 5px solid #d9534f; background-color: #fdf7f7; color: #a94442;">
<strong>🛑 ATENÇÃO:</strong> rode as células Python abaixo. Elas imprimem pergunta, SQL já preenchido e Usage Guidance. O texto da célula SQL <em>não</em> troca <code>${widget}</code> sozinho.
</div>


---
**Exemplo 1** — custo e R$/t por modo (metric view)


In [0]:
nome_catalogo = dbutils.widgets.get("nome_catalogo").strip()
nome_schema = dbutils.widgets.get("nome_schema").strip()
seu_prefixo = dbutils.widgets.get("seu_prefixo").strip()

pergunta_1 = "Quais modos de transporte têm maior custo total e qual o custo por tonelada de cada um?"
guidance_1 = (
    "Use a metric view de logística. "
    "MEASURE(custo_total) e MEASURE(custo_por_tonelada). "
    "Agrupe por modo_transporte. Nunca some ocupacao_media."
)
sql_exemplo_1 = f"""
SELECT
  modo_transporte,
  ROUND(MEASURE(`custo_total`), 2) AS custo_total,
  ROUND(MEASURE(`volume_total`), 2) AS volume_total,
  ROUND(MEASURE(`custo_por_tonelada`), 2) AS custo_por_tonelada,
  ROUND(MEASURE(`ocupacao_media`), 2) AS ocupacao_media
FROM `{nome_catalogo}`.`{nome_schema}`.`{seu_prefixo}_mv_logistica`
GROUP BY ALL
ORDER BY custo_total DESC
""".strip()

print("=== What question does this query answer? ===")
print(pergunta_1)
print()
print("=== SQL ===")
print(sql_exemplo_1)
print()
print("=== Usage Guidance ===")
print(guidance_1)

display(spark.sql(sql_exemplo_1))


---
**Exemplo 2** — custo e OTIF por estado de origem


In [0]:
nome_catalogo = dbutils.widgets.get("nome_catalogo").strip()
nome_schema = dbutils.widgets.get("nome_schema").strip()
seu_prefixo = dbutils.widgets.get("seu_prefixo").strip()

pergunta_2 = "Quais estados de origem concentram custo logístico e onde o OTIF está pior?"
guidance_2 = (
    "Use a metric view. GROUP BY estado_origem. "
    "Custo é MEASURE(custo_total); OTIF é MEASURE(otif), nunca AVG de um percentual já agregado."
)
sql_exemplo_2 = f"""
SELECT
  estado_origem,
  ROUND(MEASURE(`custo_total`), 2) AS custo_total,
  ROUND(MEASURE(`volume_total`), 2) AS volume_total,
  ROUND(MEASURE(`otif`), 2) AS otif,
  MEASURE(`entregas_atrasadas`) AS entregas_atrasadas
FROM `{nome_catalogo}`.`{nome_schema}`.`{seu_prefixo}_mv_logistica`
GROUP BY ALL
ORDER BY custo_total DESC
""".strip()

print("=== What question does this query answer? ===")
print(pergunta_2)
print()
print("=== SQL ===")
print(sql_exemplo_2)
print()
print("=== Usage Guidance ===")
print(guidance_2)

display(spark.sql(sql_exemplo_2))


---
**Exemplo 3** — distância final das rotas (tabela geo, não a metric view)


In [0]:
nome_catalogo = dbutils.widgets.get("nome_catalogo").strip()
nome_schema = dbutils.widgets.get("nome_schema").strip()
seu_prefixo = dbutils.widgets.get("seu_prefixo").strip()

pergunta_3 = "Mostre as rotas ativas, o modo e a distância final de cada uma."
guidance_3 = (
    "Use rotas_entrega, não a metric view. "
    "Agrupe por rota_id. Distância final = MAX(distancia_acumulada_km). "
    "Ordene pontos por ordem_ponto quando a pergunta for de trajeto."
)
sql_exemplo_3 = f'''
SELECT
  rota_id,
  modo_transporte,
  operadora,
  ANY_VALUE(hub_origem_id) AS hub_origem_id,
  ANY_VALUE(hub_destino_id) AS hub_destino_id,
  MAX(distancia_acumulada_km) AS distancia_final_km
FROM `{nome_catalogo}`.`{nome_schema}`.`{seu_prefixo}_rotas_entrega`
GROUP BY rota_id, modo_transporte, operadora
ORDER BY distancia_final_km DESC
'''.strip()

print("=== What question does this query answer? ===")
print(pergunta_3)
print()
print("=== SQL ===")
print(sql_exemplo_3)
print()
print("=== Usage Guidance ===")
print(guidance_3)

display(spark.sql(sql_exemplo_3))


---
### Laboratório Genie Agent

Inspecione o SQL antes de aceitar a resposta.

Perguntas factuais:

1. `Quais modos de transporte têm maior custo total?`
2. `Qual o custo por tonelada por operadora?`
3. `Compare OTIF e ocupação média de Caminhão, Barco e Ferrovia.`
4. `Quais corredores origem–destino mais caros em R$?`
5. `Como está a eficiência?`

Na pergunta 5, o Agent deve pedir esclarecimento (`custo_por_tonelada`, `ocupacao_media` ou `percentual_volume_modal_alternativo`).

Perguntas hipotéticas / decisão:

6. `Se migrarmos 20% do volume de caminhão do Centro-Oeste para ferrovia, qual o impacto estimado em custo total e R$/t?`
7. `Se priorizarmos barco e balsa na região Norte, o que acontece com prazo médio e OTIF?`
8. `Onde reduzir custo sem perder demanda: quais modos estão caros em R$/t com ocupação abaixo de 75%?`
9. `Se eliminarmos entregas atrasadas nos três estados de origem mais caros, quanto de custo e volume está em risco?`
10. `Qual hub deveria ganhar capacidade se o objetivo for atender mais demanda no menor custo?`

Perguntas geográficas (o Agent deve responder com mapa sempre que possível):

11. `Gere um mapa do Brasil mostrando o volume total transportado por estado de origem`
12. `Quais hubs ficam a menos de 500 km do hub de Vitória?`
13. `Crie um mapa de calor por estado mostrando a taxa de OTIF (entregas no prazo)`
14. `Mapeie o custo total de frete por estado de destino em escala de cores`
15. `Mostra a distribuição geográfica dos gastos logísticos`
16. `Crie um mapa do percentual de entregas atrasadas por estado de destino`


---
## Módulo 5 — Genie One

1. No topo da página, clique no botão com nove bolinhas.
2. Abra o **Genie One**.
3. Faça as **mesmas perguntas** do laboratório do Genie Agent, inclusive as geográficas (mapas por estado e proximidade de hubs).

Compare com a sala curada:

| | Genie Agent | Genie One |
|---|---|---|
| Fontes | Só as que você adicionou + metric view | Mais amplo no workspace |
| Instruções | As suas, em português | Menos contexto de negócio |
| Exemplos SQL | Os três que você salvou | Pode não conhecer R$/t como razão de somas |

Pergunta boa para contrastar: `Como está a eficiência?`  
No Agent curado, deve pedir esclarecimento. No One, pode chutar uma métrica.

Feche a sessão no trade-off do cliente: **atender demanda, menor custo, sem quebrar OTIF**, com KPI governado na metric view e mapas no AI/BI.
